In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## Step 1: Load the Dataset and Report Shape, Columns, and Dtypes

In [ ]:
df = pd.read_csv(r"C:\Users\sadee\OneDrive\Desktop\The Main Folder\The Main Folder\Week1\Day4\dirty_cafe_sales.csv")

print("Shape (rows, columns):", df.shape)
print()
print("Columns:", list(df.columns))
print()
print("Dtypes:")
print(df.dtypes)

Shape (rows, columns): (10000, 8)

Columns: ['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Dtypes:
Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object


In [ ]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


## Step 2: Count Missing Values per Column and Handle Them



In [ ]:
# Count of real missing values (NaN) per column, before any processing
print("Missing values per column (raw NaN only):")
print(df.isna().sum())

Missing values per column (raw NaN only):
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


In [ ]:
df_clean = df.replace(['UNKNOWN', 'ERROR'], np.nan)

print("Missing values per column (after treating UNKNOWN/ERROR as missing):")
print(df_clean.isna().sum())

Missing values per column (after treating UNKNOWN/ERROR as missing):
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


### Decision and Justification



In [ ]:
# Convert numeric columns to actual numeric dtypes after cleaning out invalid text values
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Convert the date column to datetime
df_clean['Transaction Date'] = pd.to_datetime(df_clean['Transaction Date'], errors='coerce')

# 1) Drop rows missing the core numeric columns or the date
df_clean = df_clean.dropna(subset=['Quantity', 'Price Per Unit', 'Total Spent', 'Transaction Date'])

# 2) Impute Item with a clear placeholder label
df_clean['Item'] = df_clean['Item'].fillna('Unknown Item')

# 3) Impute Payment Method and Location with "Unknown"
df_clean['Payment Method'] = df_clean['Payment Method'].fillna('Unknown')
df_clean['Location'] = df_clean['Location'].fillna('Unknown')

print("Shape after cleaning:", df_clean.shape)
print()
print("Remaining missing values per column:")
print(df_clean.isna().sum())

Shape after cleaning: (8159, 8)

Remaining missing values per column:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


In [ ]:
df_clean.dtypes

Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

## Step 3: Filter the Data to a Meaningful Subset


In [ ]:
avg_spent = df_clean['Total Spent'].mean()
print(f"Average transaction value (Total Spent): {avg_spent:.2f}")

high_value = df_clean[df_clean['Total Spent'] > avg_spent]
print(f"Number of high-value transactions: {len(high_value)} out of {len(df_clean)} "
      f"({len(high_value) / len(df_clean):.1%})")

high_value.head(10)

Average transaction value (Total Spent): 8.91
Number of high-value transactions: 3539 out of 8159 (43.4%)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,Unknown,2023-03-31
6,TXN_4433211,Unknown Item,3.0,3.0,9.0,Unknown,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,Unknown,2023-10-28
8,TXN_4717867,Unknown Item,5.0,3.0,15.0,Unknown,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5.0,4.0,20.0,Unknown,In-store,2023-12-31
10,TXN_2548360,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-11-07
15,TXN_2847255,Salad,3.0,5.0,15.0,Credit Card,In-store,2023-11-15
18,TXN_8876618,Cake,5.0,3.0,15.0,Cash,Unknown,2023-03-25


In [ ]:
print("Most common items in high-value transactions:")
print(high_value['Item'].value_counts().head())
print()
print("Most common payment methods in high-value transactions:")
print(high_value['Payment Method'].value_counts())
print()
print("Most common locations in high-value transactions:")
print(high_value['Location'].value_counts())

Most common items in high-value transactions:
Item
Salad       752
Juice       571
Cake        562
Sandwich    552
Smoothie    549
Name: count, dtype: int64

Most common payment methods in high-value transactions:
Payment Method
Unknown           1063
Credit Card        829
Digital Wallet     829
Cash               818
Name: count, dtype: int64

Most common locations in high-value transactions:
Location
Unknown     1405
In-store    1082
Takeaway    1052
Name: count, dtype: int64


## Step 4: Use `groupby` to Compute an Aggregate Statistic per Category

We compute the **average transaction value (`Total Spent`) per item type (`Item`)**, then sort in descending order to see which items generate the highest spend per transaction.

In [ ]:
item_avg_spent = (
    df_clean.groupby('Item')['Total Spent']
    .agg(['mean', 'sum', 'count'])
    .rename(columns={'mean': 'Avg Total Spent', 'sum': 'Total Revenue', 'count': 'Num Transactions'})
    .sort_values('Avg Total Spent', ascending=False)
)

item_avg_spent

,Avg Total Spent,Total Revenue,Num Transactions
Item,,,
Salad,15.149413,14195.0,937
Smoothie,12.261261,10888.0,888
Sandwich,12.013115,10992.0,915
Cake,9.095440,8577.0,943
Juice,8.879630,8631.0,972
Unknown Item,8.853282,6879.0,777
Coffee,6.125903,5936.0,969
Tea,4.534208,3976.5,877
Cookie,2.985244,2630.0,881
